# 01 — Dataset Inspection

> **Objective:** Load the raw CSV, profile structure and quality, parse `last_updated`, assess temporal granularity and geographic coverage.

---

## 1. Setup

Import libraries for data loading, inspection, and formatted output.

In [18]:
import pandas as pd
from IPython.display import Markdown, display

## 2. Load Dataset

Read `GlobalWeatherRepository.csv` from `data/raw/` and inspect shape, sample rows, dtypes, and missing values.

In [19]:
# 1) Load dataset and inspect high-level structure
raw_data = pd.read_csv('../data/raw/GlobalWeatherRepository.csv')

dataset_shape = raw_data.shape
sample_rows = raw_data.head()
dtypes = raw_data.dtypes
null_counts = raw_data.isna().sum().sort_values(ascending=False)

print('Shape:', dataset_shape)
display(sample_rows)
display(dtypes)
display(null_counts.head(10))

Shape: (133318, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


country                             str
location_name                       str
latitude                        float64
longitude                       float64
timezone                            str
last_updated_epoch                int64
last_updated                        str
temperature_celsius             float64
temperature_fahrenheit          float64
condition_text                      str
wind_mph                        float64
wind_kph                        float64
wind_degree                       int64
wind_direction                      str
pressure_mb                     float64
pressure_in                     float64
precip_mm                       float64
precip_in                       float64
humidity                          int64
cloud                             int64
feels_like_celsius              float64
feels_like_fahrenheit           float64
visibility_km                   float64
visibility_miles                float64
uv_index                        float64


country                   0
location_name             0
latitude                  0
longitude                 0
timezone                  0
last_updated_epoch        0
last_updated              0
temperature_celsius       0
temperature_fahrenheit    0
condition_text            0
dtype: int64

## 3. Parse Temporal Column

Convert `last_updated` to datetime, validate parsing, and set as time index.

In [20]:
# 2) Parse temporal column and build time-indexed frame
df = raw_data.copy()
df['last_updated'] = pd.to_datetime(df['last_updated'], errors='coerce')

invalid_timestamps = int(df['last_updated'].isna().sum())

df = df.sort_values(by='last_updated').set_index('last_updated')
start_ts, end_ts = df.index.min(), df.index.max()

print('Datetime dtype:', df.index.dtype)
print('Invalid timestamps:', invalid_timestamps)
print('Time window:', start_ts, '->', end_ts)

Datetime dtype: datetime64[us]
Invalid timestamps: 0
Time window: 2024-05-16 01:45:00 -> 2026-04-03 19:30:00


## 4. Temporal Granularity

Compute successive time differences to identify dominant spacing and duplicate timestamps.

In [21]:
# 3) Temporal granularity diagnostics
time_diff = df.index.to_series().diff().dropna()
granularity_counts = time_diff.value_counts()
dominant_granularity = granularity_counts.index[0] if not granularity_counts.empty else None
same_timestamp_count = int((time_diff == pd.Timedelta(0)).sum())

print('Dominant granularity:', dominant_granularity)
print('Rows sharing same timestamp:', same_timestamp_count)
display(time_diff.describe())
display(granularity_counts.head(10))

Dominant granularity: 0 days 00:00:00
Rows sharing same timestamp: 111043


count                    133317
mean     0 days 00:07:25.709849
std      0 days 00:24:34.611108
min             0 days 00:00:00
25%             0 days 00:00:00
50%             0 days 00:00:00
75%             0 days 00:00:00
max             1 days 15:45:00
Name: last_updated, dtype: object

last_updated
0 days 00:00:00    111043
0 days 00:15:00      7495
0 days 01:00:00      6145
0 days 00:30:00      4477
0 days 00:45:00      2523
0 days 02:00:00       665
0 days 01:45:00       198
0 days 04:00:00       158
0 days 03:45:00       119
0 days 03:00:00       102
Name: count, dtype: int64

## 5. Geographic Coverage

Summarize unique countries/continents and missing-value rates per column.

In [22]:
# 4) Geographic coverage + null profile
country_col = 'country' if 'country' in df.columns else None
continent_col = 'continent' if 'continent' in df.columns else None

n_countries = int(df[country_col].nunique(dropna=True)) if country_col else 0
n_continents = int(df[continent_col].nunique(dropna=True)) if continent_col else 0

top_countries = df[country_col].value_counts(dropna=False).head(10) if country_col else pd.Series(dtype='int64')
top_continents = df[continent_col].value_counts(dropna=False).head(10) if continent_col else pd.Series(dtype='int64')

null_pct = ((df.isna().sum() / len(df)) * 100).sort_values(ascending=False)

print('Unique countries:', n_countries)
print('Unique continents:', n_continents)
display(top_countries)
display(top_continents)
display(null_pct.head(10).round(2))

Unique countries: 211
Unique continents: 0


country
Bulgaria      1583
Indonesia     1371
Thailand      1370
Sudan         1366
Turkey        1366
Bolivia       1359
Iran          1332
Belgium       1290
Madagascar    1143
Vietnam       1131
Name: count, dtype: int64

Series([], dtype: int64)

country                   0.0
location_name             0.0
latitude                  0.0
longitude                 0.0
timezone                  0.0
last_updated_epoch        0.0
temperature_celsius       0.0
temperature_fahrenheit    0.0
condition_text            0.0
wind_mph                  0.0
dtype: float64

## 6. Executive Summary

Automated Markdown report with key findings from the inspection.

In [23]:
# 5) Executive summary — rendered as Markdown
top_null_counts = null_counts.head(5)
top_null_lines = "\n".join(f"- `{col}`: {int(val):,}" for col, val in top_null_counts.items())

summary_md = f"""
## Dataset inspection summary

- **Shape:** {dataset_shape[0]:,} rows × {dataset_shape[1]} columns.
- **Time window (`last_updated`):** {start_ts} → {end_ts}.
- **Datetime parsing issues:** {invalid_timestamps} invalid row(s).
- **Dominant temporal granularity:** `{dominant_granularity}`.
- **Duplicate timestamp transitions (0 diff):** {same_timestamp_count:,}.
- **Geographic coverage:** {n_countries} countries; {n_continents} distinct continent values (if the column exists).

### Top 5 columns by missing values (count)
{top_null_lines}

### Initial observations
1. Timestamps are not strictly regular; many rows share the same `last_updated` (multi-location snapshots).
2. Country distribution may be imbalanced — validate before modeling or aggregate by region.
3. Use missing-value patterns from above to drive imputation order in preprocessing.
"""

display(Markdown(summary_md))



## Dataset inspection summary

- **Shape:** 133,318 rows × 41 columns.
- **Time window (`last_updated`):** 2024-05-16 01:45:00 → 2026-04-03 19:30:00.
- **Datetime parsing issues:** 0 invalid row(s).
- **Dominant temporal granularity:** `0 days 00:00:00`.
- **Duplicate timestamp transitions (0 diff):** 111,043.
- **Geographic coverage:** 211 countries; 0 distinct continent values (if the column exists).

### Top 5 columns by missing values (count)
- `country`: 0
- `location_name`: 0
- `latitude`: 0
- `longitude`: 0
- `timezone`: 0

### Initial observations
1. Timestamps are not strictly regular; many rows share the same `last_updated` (multi-location snapshots).
2. Country distribution may be imbalanced — validate before modeling or aggregate by region.
3. Use missing-value patterns from above to drive imputation order in preprocessing.
